In [ ]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Load env variables from .env file
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path(r"C:/AI-Training/Anthropic-learning/My-Anthropic-Training")
dotenv_path = workspace_root / ".env"
load_dotenv(dotenv_path, override=True)
print("API key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("dotenv path:", dotenv_path)

API key loaded: True
dotenv path: C:\AI-Training\Anthropic-learning\My-Anthropic-Training\.env


In [2]:
# create an API client
import os
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

workspace_root = Path(r"C:/AI-Training/Anthropic-learning/My-Anthropic-Training")
dotenv_path = workspace_root / ".env"
load_dotenv(dotenv_path, override=True)

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is not set. Add it to your .env file.")
if len(api_key) < 20:
    raise ValueError("ANTHROPIC_API_KEY looks too short to be valid.")

client = Anthropic(api_key=api_key)
model = "claude-sonnet-4-5"
print("Client ready")

Client ready


In [3]:
from anthropic import AuthenticationError

try:
    message = client.messages.create(
        model=model,
        max_tokens=100,
        messages=[{"role": "user", "content": "What is Quantum Computing? Answer in one sentence."}],
    )
    print(message.content[0].text)
except AuthenticationError as exc:
    raise RuntimeError(
        "Anthropic authentication failed. The key in .env is invalid, expired, or missing. "
        "Replace ANTHROPIC_API_KEY with a valid key from the Anthropic Console."
    ) from exc

Quantum computing is a type of computation that harnesses the properties of quantum mechanics—such as superposition and entanglement—to process information in ways that can solve certain complex problems much faster than classical computers.


In [4]:
import os
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

workspace_root = Path(r"C:/AI-Training/Anthropic-learning/My-Anthropic-Training")
dotenv_path = workspace_root / ".env"
load_dotenv(dotenv_path, override=True)

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is not set. Add it to your .env file.")
if len(api_key) < 20:
    raise ValueError("ANTHROPIC_API_KEY looks too short to be valid.")

client = Anthropic(api_key=api_key)
model = "claude-sonnet-4-5"
print("Client ready")

Client ready


In [15]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text


In [16]:
# Solution hidden !
# Make an initial message list of messages
messages = []

# Use a 'while True' loop to run the chatbot forever

while True:
    # Get user input
    user_input = input("> ")
    print(">", user_input)
    
    # Add user input to the list of messages
    add_user_message(messages, user_input)
    
    # Call Claude with 'chat' function and get the response
    answer = chat(messages)
    
    # Add generated text to the list of messages
    add_assistant_message(messages, answer)
    
    # Print the generated text
    print("------")
    print(answer)
    print("------")

#

> 8+9=?
------
8 + 9 = 17
------
> 


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: user messages must have non-empty content'}, 'request_id': 'req_011Ce45wkE9uqLu5Yy5bbwK5'}

In [8]:
messages = []

system = """""
you are a patient math tutor.
Do not directly answer a student's question.
Guide them to a solution step by step.
"""""
add_user_message(messages, "How do I Solve 5x+3=2 for x?")
answer = chat(messages, system=system)

answer

'Great question! Let\'s work through this together step by step.\n\nFirst, let me ask you: **What do you think our goal is when we "solve for x"?** What do we want x to look like when we\'re done?\n\nOnce you answer that, I\'ll help you figure out what steps we need to take to get there! \n\n*Hint: Think about what it means to have x "by itself" or "isolated."*'

In [10]:
messages = []

add_user_message(messages, "write a Python function that checks a string for duplicate characters.")
answer = chat(messages, system="You are Python Engineer who writes very concise code.")

answer

'```python\ndef has_duplicates(s: str) -> bool:\n    return len(s) != len(set(s))\n```\n\n**Usage:**\n```python\nhas_duplicates("hello")   # True (duplicate \'l\')\nhas_duplicates("world")   # True (duplicate \'o\' and \'l\')\nhas_duplicates("python")  # False\n```\n\n**Alternative** (if you need to find which characters are duplicated):\n```python\ndef find_duplicates(s: str) -> set:\n    return {c for c in s if s.count(c) > 1}\n```'

In [17]:
from anthropic import Anthropic

client = Anthropic()
messages = []

def ask(user_text):
    messages.append({"role": "user", "content": user_text})
    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=300,
        messages=messages,
    )

    text_blocks = [b.text for b in response.content if getattr(b, "type", None) == "text"]

    if not text_blocks:
        block_types = [getattr(b, "type", None) for b in response.content]
        raise ValueError(f"No text blocks returned from Anthropic. Response blocks: {block_types}")

    reply = "".join(text_blocks)
    messages.append({"role": "assistant", "content": reply})
    print(f"Claude: {reply}\n")

ask("My name is Priya and I work on the Nimbus migration project. Remember that.")
ask("What's my name and what project am I working on?")

Claude: I'll remember that your name is Priya and you're working on the Nimbus migration project.

To make sure I'm actually useful to you going forward, it'd help to know a bit more:

- What kind of migration is Nimbus — a cloud platform switch, a database migration, an internal tool being replaced, something else?
- What's your role on it (leading it, doing the technical work, coordinating people, etc.)?
- What do you want help with — technical troubleshooting, planning, writing docs/status updates, something else?

One thing worth knowing: I don't have persistent memory across separate conversations unless this platform has a memory feature turned on for your account. Within *this* conversation, I've got it. If you start a new chat later, you may need to remind me. If you tell me more about the project now, I can be a lot more targeted in helping you rather than giving generic advice.

Claude: Your name is Priya, and you work on the Nimbus migration project.

A quick caveat, since i